In [57]:
import pandas as pd
import fasttext
import fasttext.util
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load dataset
df = pd.read_csv("universities.csv")

# Drop rows with missing course info
df.dropna(subset=['MS Courses Offered'], inplace=True)

In [18]:
import joblib

In [56]:
model = joblib.load("xgb_qs_model_1.pkl")

In [ ]:
# Load pre-trained fasttext word vectors
fasttext.util.download_model('en', if_exists='ignore')  # downloads 'cc.en.300.bin'

 (100.00%) [==================================================>]                                                 ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]> 

In [22]:
ft = fasttext.load_model('cc.en.300.bin')

# Helper function to get FastText sentence embedding
def get_sentence_vector(text):
    return ft.get_sentence_vector(text)

# Generate embeddings for each course offering
df['course_vector'] = df['MS Courses Offered'].apply(lambda x: get_sentence_vector(str(x)))

In [61]:
import numpy as np

def parse_vector(vec_str):
    # Remove brackets and split on spaces
    return np.array([float(x) for x in vec_str.strip("[]").split()])

df['course_vector'] = df['course_vector'].apply(parse_vector)


In [72]:
from sklearn.metrics.pairwise import cosine_similarity

df['course_similarity'] = df['course_vector'].apply(
    lambda vec: cosine_similarity([user_vec], [vec])[0][0]
)


In [73]:
# -------- USER INPUT SECTION --------
user_course = input("Enter your desired course (e.g., 'Computer Engineering'): ").strip()
user_vec = get_sentence_vector(user_course)

# -------- Helper Function to Get Best Matching Course --------
def get_best_matching_course(courses_str):
    courses = [c.strip() for c in courses_str.split(',')]
    best_course = None
    best_score = -1
    for course in courses:
        vec = get_sentence_vector(course)
        score = cosine_similarity([user_vec], [vec])[0][0]
        if score > best_score:
            best_score = score
            best_course = course
    return best_course, best_score

# -------- Apply to DataFrame --------
df[['best_course_match', 'best_course_score']] = df['MS Courses Offered'].apply(
    lambda x: pd.Series(get_best_matching_course(x))
)

# -------- Sort and Display --------
df_sorted = df.sort_values(by='best_course_score', ascending=False).head(5)
top_univ = df_sorted.iloc[0]

# -------- Output --------
print("\n🔍 You searched for:", user_course)
print(f"💡 Best Matching Course Found: {top_univ['best_course_match']} (from {top_univ['University Name']})")

print("\n🎯 Top University Matches for Your Course:")
print("-" * 70)
for idx, row in df_sorted.iterrows():
    print(f"🏫 University: {row['University Name']}")
    print(f"🎓 Courses Offered: {row['MS Courses Offered']}")
    print(f"🤝 Closest Course Match: {row['best_course_match']}")
    print(f"📈 Match Score: {round(row['best_course_score'] * 100, 2)}%")
    print("-" * 70)



🔍 You searched for: mechanical engineering\
💡 Best Matching Course Found: MS in Mechanical Engineering (from Alabama A & M University)

🎯 Top University Matches for Your Course:
----------------------------------------------------------------------
🏫 University: Alabama A & M University
🎓 Courses Offered: MS in Biology, MS in Education, MS in Finance, MS in Marketing, MS in Mechanical Engineering
🤝 Closest Course Match: MS in Mechanical Engineering
📈 Match Score: 44.38%
----------------------------------------------------------------------
🏫 University: The University Of Texas At Dallas
🎓 Courses Offered: MS in Biology, MS in Civil Engineering, MS in Accounting, MS in Finance, MS in Computer Science, MS in Business Analytics, MS in Cybersecurity, MS in Mechanical Engineering, MBA, MS in Information Systems, MS in Data Science, MS in Business Analysis
🤝 Closest Course Match: MS in Mechanical Engineering
📈 Match Score: 44.38%
-------------------------------------------------------------

COMBINED MODEL TESTING

In [8]:
import pandas as pd
import numpy as np
import joblib
import fasttext
import fasttext.util
from sklearn.metrics.pairwise import cosine_similarity

In [74]:
df = pd.read_csv('universities.csv')

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity

# Load everything
df = pd.read_csv("universities.csv")
model = joblib.load("xgb_qs_model_1.pkl")

# Ensure numeric
df['QS Overall Score'] = pd.to_numeric(df['QS Overall Score'], errors='coerce')
df.dropna(subset=['QS Overall Score'], inplace=True)

# --- COURSE MAPPING DICTIONARY ---
course_aliases = {
    "MS in Biomedical Sciences": [
        "molecular biology", "biochemistry", "genetics", "microbiology",
        "clinical research", "human physiology", "immunology", "biotechnology"
    ],
    "MS in Business Analysis": [
        "business analytics", "decision sciences", "operations research", "quantitative business", "enterprise analytics"
    ],
    "MS in Business Analytics": [
        "data analytics", "predictive modeling", "marketing analytics", "analytics in finance", "supply chain analytics"
    ],
    "MS in Chemistry": [
        "analytical chemistry", "organic chemistry", "physical chemistry", "polymer science", "industrial chemistry"
    ],
    "MS in Civil Engineering": [
        "structural engineering", "urban planning", "environmental engineering", "geotechnical engineering",
        "water resources engineering", "architecture", "construction management"
    ],
    "MS in Computer Science": [
        "cloud computing", "artificial intelligence", "machine learning", "web development", "software engineering",
        "frontend development", "backend development", "app development", "full-stack development",
        "computer vision", "natural language processing", "deep learning"
    ],
    "MS in Criminal Justice": [
        "law enforcement", "forensic science", "crime scene investigation", "homeland security", "criminology"
    ],
    "MS in Cybersecurity": [
        "information security", "ethical hacking", "digital forensics", "network security", "penetration testing"
    ],
    "MS in Data Science": [
        "data mining", "statistical modeling", "big data", "data engineering", "applied statistics",
        "business intelligence"
    ],
    "MS in Education": [
        "curriculum design", "educational technology", "special education", "teacher education",
        "instructional design", "pedagogy"
    ],
    "MS in Electrical Engineering": [
        "electronics", "signal processing", "power systems", "telecommunications", "embedded systems",
        "robotics", "semiconductors"
    ],
    "MS in Finance": [
        "investment banking", "financial engineering", "financial analytics", "quantitative finance",
        "corporate finance", "fintech"
    ],
    "MS in Information Systems": [
        "MIS", "IT management", "enterprise systems", "ERP systems", "database management", "systems analysis"
    ],
    "MS in Management": [
        "general management", "strategic leadership", "organizational behavior", "operations management",
        "international business"
    ],
    "MS in Marketing": [
        "digital marketing", "brand management", "market research", "consumer psychology", "advertising strategy"
    ],
    "MS in Mechanical Engineering": [
        "thermodynamics", "fluid mechanics", "automotive engineering", "aerospace", "manufacturing systems"
    ],
    "MS in Nursing": [
        "nurse practitioner", "clinical nursing", "healthcare informatics", "psychiatric nursing"
    ],
    "MS in Public Health": [
        "epidemiology", "health policy", "biostatistics", "global health", "community health"
    ]
}

# --- HELPER TO MAP ALIAS TO STANDARDIZED COURSE ---
def map_to_standard_course(user_input):
    user_input_lower = user_input.lower()
    for standard, aliases in course_aliases.items():
        if user_input_lower == standard.lower():
            return standard
        for alias in aliases:
            if alias in user_input_lower:
                return standard
    return user_input  # fallback to original if no match

# ---- USER INPUT SECTION ----
user_course_raw = input("Enter your desired course or area of interest (e.g., 'Cloud Computing'): ").strip()
standardized_course = map_to_standard_course(user_course_raw)
user_vec = get_sentence_vector(standardized_course)

tuition = float(input("Enter your estimated tuition budget (in USD): "))
gpa = float(input("Enter your GPA (on a 4.0 scale): "))
gre = float(input("Enter your GRE score (out of 340): "))

# ---- Predict QS Score ----
user_input = pd.DataFrame({
    'Out-of-State Tuition': [tuition],
    'Average GPA': [gpa],
    'Average GRE Score': [gre]
})

predicted_qs = model.predict(user_input)[0]
df['qs_diff'] = abs(df['QS Overall Score'] - predicted_qs)

# ---- Semantic Course Matching ----
def get_best_matching_course(course_list, user_vec):
    courses = [c.strip() for c in course_list.split(',')]
    sims = [cosine_similarity([user_vec], [get_sentence_vector(c)])[0][0] for c in courses]
    return courses[np.argmax(sims)], max(sims)

df[['best_course_match', 'course_similarity']] = df.apply(
    lambda row: pd.Series(get_best_matching_course(row['MS Courses Offered'], user_vec)),
    axis=1
)

# ---- Sort and Recommend ----
top_matches = df.sort_values(by=['qs_diff', 'course_similarity'], ascending=[True, False]).head(10)

# ---- Output ----
print(f"\n🔍 You searched for: {user_course_raw}")
print(f"📘 You can go for {standardized_course}")
print(f"💡 Best Matching Course Found: {top_matches.iloc[0]['best_course_match']} (from {top_matches.iloc[0]['University Name']})")

print("\n🎯 Top 10 University Matches:")
print("-" * 70)
for _, row in top_matches.iterrows():
    print(f"🏫 University: {row['University Name']}")
    print(f"🎓 Courses Offered: {row['MS Courses Offered']}")
    print(f"💡 Best Matching Course: {row['best_course_match']}")
    print(f"💰 Tuition: ${row['Out-of-State Tuition']} | Required GPA: {row['Average GPA']} | Required GRE: {row['Average GRE Score']}")
    print(f"📈 Predicted QS Score Match: {round(predicted_qs, 2)} | Uni QS: {row['QS Overall Score']}")
    print(f"🤝 Course Match Score: {round(row['course_similarity'] * 100, 2)}%")
    print("-" * 70)



🔍 You searched for: architecture
📘 Interpreted as: MS in Civil Engineering
💡 Best Matching Course Found: MS in Civil Engineering (from Michigan State University)

🎯 Top 10 University Matches:
----------------------------------------------------------------------
🏫 University: Michigan State University
🎓 Courses Offered: MS in Civil Engineering, MS in Mechanical Engineering, MS in Finance, MS in Computer Science, MS in Cybersecurity, MBA, MS in Information Systems, MS in Data Science, MS in Electrical Engineering, MS in Business Analysis
💡 Best Matching Course: MS in Civil Engineering
💰 Tuition: $41958.0 | GPA: 2.6 | GRE: 275.0
📈 Predicted QS Score Match: 50.75 | Uni QS: 51.9
🤝 Course Match Score: 100.0%
----------------------------------------------------------------------
🏫 University: University Of California Santa Barbara
🎓 Courses Offered: MS in Accounting, MS in Finance, MS in Computer Science, MS in Cybersecurity, MS in Mechanical Engineering, MS in Marketing, MS in Information 

In [ ]:
def get_best_matching_course(course_list, user_vec, threshold=0.5):  # Increase the threshold
    courses = [c.strip() for c in course_list.split(',')]
    sims = [cosine_similarity([user_vec], [get_sentence_vector(c)])[0][0] for c in courses]
    max_sim = max(sims)
    best_course = courses[np.argmax(sims)]
    
    if max_sim < threshold:  # Only consider strong matches above the threshold
        return None, max_sim

    return best_course, max_sim

# ---- FUNCTION TO GET UNIVERSITY RECOMMENDATION BASED ON USER INPUT ----
def recommend_university(user_course_raw, tuition, gpa, gre):
    # First, try to use FastText for matching
    user_vec = get_sentence_vector(user_course_raw)
    
    # Predict QS Score based on user inputs (tuition, GPA, GRE)
    user_input = pd.DataFrame({
        'Out-of-State Tuition': [tuition],
        'Average GPA': [gpa],
        'Average GRE Score': [gre]
    })

    predicted_qs = model.predict(user_input)[0]
    df['qs_diff'] = abs(df['QS Overall Score'] - predicted_qs)

    # Calculate best matching course for each university in the dataset using FastText
    df[['best_course_match', 'course_similarity']] = df.apply(
        lambda row: pd.Series(get_best_matching_course(row['MS Courses Offered'], user_vec)),
        axis=1
    )

    # If no good match found with FastText, fall back to course mapping
    df['course_match_found'] = df['best_course_match'].notnull()

    # If FastText didn't give a good match, use the course alias dictionary for fallback
    df['best_course_match'] = df.apply(
        lambda row: map_to_standard_course(user_course_raw) if not row['course_match_found'] else row['best_course_match'],
        axis=1
    )

    # ---- Sort and Recommend ----
    top_matches = df.sort_values(by=['qs_diff', 'course_similarity'], ascending=[True, False]).head(10)

    # ---- Output ----
    print(f"\n🔍 You searched for: {user_course_raw}")
    
    if top_matches.iloc[0]['best_course_match'] is None:
        print("❌ No matching course found!")
        return None  # If no match found
    
    print(f"📘 Best Matching Course Found: {top_matches.iloc[0]['best_course_match']}")
    
    print("\n🎯 Top 10 University Matches:")
    print("-" * 70)
    for _, row in top_matches.iterrows():
        print(f"🏫 University: {row['University Name']}")
        print(f"🎓 Courses Offered: {row['MS Courses Offered']}")
        print(f"💡 Best Matching Course: {row['best_course_match']}")
        print(f"💰 Tuition: ${row['Out-of-State Tuition']} | Required GPA: {row['Average GPA']} | Required GRE: {row['Average GRE Score']}")
        print(f"📈 Predicted QS Score Match: {round(predicted_qs, 2)} | Uni QS: {row['QS Overall Score']}")
        print(f"🤝 Course Match Score: {round(row['course_similarity'] * 100, 2)}%")
        print("-" * 70)



In [126]:
# Example call:
recommend_university('biotechnology', 18000, 3.9, 335)


🔍 You searched for: biotechnology
📘 Best Matching Course Found: MS in Biomedical Sciences

🎯 Top 10 University Matches:
----------------------------------------------------------------------
🏫 University: University Of California San Diego
🎓 Courses Offered: MS in Civil Engineering, MS in Mechanical Engineering, MS in Management, MS in Computer Science, MS in Business Analytics, MS in Finance, MS in Cybersecurity, MS in Information Systems, MS in Data Science, MS in Business Analysis, MS in Education
💡 Best Matching Course: MS in Biomedical Sciences
💰 Tuition: $44978.0 | Required GPA: 3.6 | Required GRE: 325.0
📈 Predicted QS Score Match: 67.58000183105469 | Uni QS: 67.1
🤝 Course Match Score: 29.78%
----------------------------------------------------------------------
🏫 University: The University Of Texas At Austin
🎓 Courses Offered: MS in Mechanical Engineering, MS in Finance, MS in Computer Science, MS in Cybersecurity, MS in Marketing, MS in Public Health, MS in Information Systems

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from fasttext import load_model  # Assuming you have a FastText model for vectorization

# Load everything
df = pd.read_csv("universities.csv")
model = joblib.load("xgb_qs_model_1.pkl")

# Load FastText model (you'll need the path to your FastText model)
fasttext_model = load_model('cc.en.300.bin')  # Replace with actual path

# Ensure numeric
df['QS Overall Score'] = pd.to_numeric(df['QS Overall Score'], errors='coerce')
df.dropna(subset=['QS Overall Score'], inplace=True)

# --- COURSE MAPPING DICTIONARY ---
course_aliases = {
    "MS in Biomedical Sciences": [
        "molecular biology", "biochemistry", "genetics", "microbiology",
        "clinical research", "human physiology", "immunology", "biotechnology"
    ],
    "MS in Business Analysis": [
        "business analytics", "decision sciences", "operations research", "quantitative business", "enterprise analytics"
    ],
    "MS in Business Analytics": [
        "data analytics", "predictive modeling", "marketing analytics", "analytics in finance", "supply chain analytics"
    ],
    "MS in Chemistry": [
        "analytical chemistry", "organic chemistry", "physical chemistry", "polymer science", "industrial chemistry"
    ],
    "MS in Civil Engineering": [
        "structural engineering", "urban planning", "environmental engineering", "geotechnical engineering",
        "water resources engineering", "architecture", "construction management"
    ],
    "MS in Computer Science": [
        "ai", "ml", "cloud computing", "artificial intelligence", "machine learning", "web development", "software engineering",
        "frontend development", "backend development", "app development", "full-stack development",
        "computer vision", "natural language processing", "deep learning"
    ],
    "MS in Criminal Justice": [
        "law enforcement", "forensic science", "crime scene investigation", "homeland security", "criminology"
    ],
    "MS in Cybersecurity": [
        "information security", "ethical hacking", "digital forensics", "network security", "penetration testing"
    ],
    "MS in Data Science": [
        "data mining", "statistical modeling", "big data", "data engineering", "applied statistics",
        "business intelligence"
    ],
    "MS in Education": [
        "curriculum design", "educational technology", "special education", "teacher education",
        "instructional design", "pedagogy"
    ],
    "MS in Electrical Engineering": [
        "electronics", "signal processing", "power systems", "telecommunications", "embedded systems",
        "robotics", "semiconductors"
    ],
    "MS in Finance": [
        "investment banking", "financial engineering", "financial analytics", "quantitative finance",
        "corporate finance", "fintech"
    ],
    "MS in Information Systems": [
        "MIS", "IT management", "enterprise systems", "ERP systems", "database management", "systems analysis"
    ],
    "MS in Management": [
        "general management", "strategic leadership", "organizational behavior", "operations management",
        "international business"
    ],
    "MS in Marketing": [
        "digital marketing", "brand management", "market research", "consumer psychology", "advertising strategy"
    ],
    "MS in Mechanical Engineering": [
        "thermodynamics", "fluid mechanics", "automotive engineering", "aerospace", "manufacturing systems"
    ],
    "MS in Nursing": [
        "nurse practitioner", "clinical nursing", "healthcare informatics", "psychiatric nursing"
    ],
    "MS in Public Health": [
        "epidemiology", "health policy", "biostatistics", "global health", "community health"
    ]
}

# --- HELPER TO MAP ALIAS TO STANDARDIZED COURSE ---
def map_to_standard_course(user_input):
    user_input_lower = user_input.lower()
    for standard, aliases in course_aliases.items():
        if user_input_lower == standard.lower():
            return standard
        for alias in aliases:
            if alias in user_input_lower:
                return standard
    return None  # Return None if no match is found

# ---- SEMANTIC COURSE MATCHING USING FastText MODEL ----
def get_sentence_vector(sentence):
    # Get FastText vector for the sentence
    return fasttext_model.get_sentence_vector(sentence)

def get_best_matching_course(course_list, user_vec, threshold=0.5):  # Adjust the threshold for FastText match
    courses = [c.strip() for c in course_list.split(',')]
    sims = [cosine_similarity([user_vec], [get_sentence_vector(c)])[0][0] for c in courses]
    max_sim = max(sims)
    best_course = courses[np.argmax(sims)]
    
    if max_sim < threshold:  # Only consider strong matches above the threshold
        return None, max_sim

    return best_course, max_sim

# ---- FUNCTION TO GET UNIVERSITY RECOMMENDATION BASED ON USER INPUT ----
def recommend_university(user_course_raw, tuition, gpa, gre):
    # First, try to use FastText for matching
    user_vec = get_sentence_vector(user_course_raw)
    
    # Predict QS Score based on user inputs (tuition, GPA, GRE)
    user_input = pd.DataFrame({
        'Out-of-State Tuition': [tuition],
        'Average GPA': [gpa],
        'Average GRE Score': [gre]
    })

    predicted_qs = model.predict(user_input)[0]
    df['qs_diff'] = abs(df['QS Overall Score'] - predicted_qs)

    # Calculate best matching course for each university in the dataset using FastText
    df[['best_course_match', 'course_similarity']] = df.apply(
        lambda row: pd.Series(get_best_matching_course(row['MS Courses Offered'], user_vec)),
        axis=1
    )

    # If no good match found with FastText, fall back to course mapping
    df['course_match_found'] = df['best_course_match'].notnull()

    # If FastText didn't give a good match, use the course alias dictionary for fallback
    df['best_course_match'] = df.apply(
        lambda row: map_to_standard_course(user_course_raw) if not row['course_match_found'] else row['best_course_match'],
        axis=1
    )

    # ---- Sort and Recommend ----
    top_matches = df.sort_values(by=['qs_diff', 'course_similarity'], ascending=[True, False]).head(10)

    # ---- Output ----
    print(f"\n🔍 You searched for: {user_course_raw}")
    
    if top_matches.iloc[0]['best_course_match'] is None:
        print("❌ No matching course found!")
        return None  # If no match found
    
    print(f"📘 Best Matching Course Found: {top_matches.iloc[0]['best_course_match']}")
    
    print("\n🎯 Top 10 University Matches:")
    print("-" * 70)
    for _, row in top_matches.iterrows():
        print(f"🏫 University: {row['University Name']}")
        print(f"🎓 Courses Offered: {row['MS Courses Offered']}")
        print(f"💡 Best Matching Course: {row['best_course_match']}")
        print(f"💰 Tuition: ${row['Out-of-State Tuition']} | Required GPA: {row['Average GPA']} | Required GRE: {row['Average GRE Score']}")
        print(f"📈 Predicted QS Score Match: {round(predicted_qs, 2)} | Uni QS: {row['QS Overall Score']}")
        print(f"🤝 Course Match Score: {round(row['course_similarity'] * 100, 2)}%")
        print("-" * 70)



In [152]:
# Example call:
recommend_university('ml', 45000, 2.9, 290)


🔍 You searched for: ml
📘 Best Matching Course Found: MS in Computer Science

🎯 Top 10 University Matches:
----------------------------------------------------------------------
🏫 University: Virginia Polytechnic Institute And State University
🎓 Courses Offered: MS in Mechanical Engineering, MS in Finance, MS in Computer Science, MS in Cybersecurity, MS in Information Systems, MS in Data Science, MS in Electrical Engineering, MS in Business Analysis
💡 Best Matching Course: MS in Computer Science
💰 Tuition: $34838.0 | Required GPA: 3.1 | Required GRE: 295.0
📈 Predicted QS Score Match: 30.31999969482422 | Uni QS: 30.1
🤝 Course Match Score: 19.62%
----------------------------------------------------------------------
🏫 University: University Of California Santa Cruz
🎓 Courses Offered: MS in Management, MS in Mechanical Engineering, MS in Accounting, MS in Business Analytics, MS in Computer Science, MS in Finance, MS in Cybersecurity, MS in Information Systems, MS in Data Science, MS in Bu